# Construction PPE Detector
A simple end-to-end notebook: check the GPU, download the public dataset, train YOLO, evaluate the test split, and export the model. In Colab choose **Runtime → Change runtime type → GPU**.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Enable GPU from Runtime > Change runtime type'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!pip install -q ultralytics==8.3.221
from ultralytics import YOLO
from pathlib import Path
import json, shutil

## Train
Ultralytics downloads the Construction-PPE dataset automatically. It contains 1,416 images split into train, validation, and test sets, with 11 PPE-related classes. YOLO11n is used because it trains quickly and is suitable for a junior portfolio project.

In [ ]:
model = YOLO('yolo11n.pt')
model.train(
    data='construction-ppe.yaml',
    epochs=40,
    imgsz=640,
    batch=16,
    device=0,
    patience=10,
    project='/content/runs',
    name='ppe-yolo11n',
    seed=42,
)

## Evaluate on the test split
These are the numbers to discuss in an interview. They come from images that were not used for training or early stopping.

In [ ]:
best_path = Path('/content/runs/ppe-yolo11n/weights/best.pt')
best_model = YOLO(best_path)
metrics = best_model.val(
    data='construction-ppe.yaml', split='test', imgsz=640, device=0, plots=True,
    project='/content/evaluation', name='test',
)
summary = {
    'precision': round(float(metrics.box.mp), 4),
    'recall': round(float(metrics.box.mr), 4),
    'mAP50': round(float(metrics.box.map50), 4),
    'mAP50_95': round(float(metrics.box.map), 4),
}
print(json.dumps(summary, indent=2))
Path('/content/evaluation/test/metrics.json').write_text(json.dumps(summary, indent=2))

## Preview predictions

In [ ]:
dataset_root = Path('/content/datasets/construction-ppe')
test_images = list((dataset_root / 'images/test').glob('*'))
assert test_images, 'Could not find test images'
best_model.predict(test_images[:6], conf=0.35, save=True, project='/content/predictions', name='examples')

## Export and download results

In [ ]:
onnx_path = best_model.export(format='onnx', imgsz=640, simplify=True)
release = Path('/content/construction-ppe-results')
release.mkdir(exist_ok=True)
shutil.copy2(best_path, release / 'best.pt')
shutil.copy2(onnx_path, release / 'best.onnx')
shutil.copy2('/content/evaluation/test/metrics.json', release / 'metrics.json')
for name in ('confusion_matrix.png', 'PR_curve.png', 'F1_curve.png'):
    source = Path('/content/evaluation/test') / name
    if source.exists(): shutil.copy2(source, release / name)
shutil.make_archive('/content/construction-ppe-results', 'zip', release)
print('Created /content/construction-ppe-results.zip')

In [ ]:
from google.colab import files
files.download('/content/construction-ppe-results.zip')